# 01 — CVM Empirical Pipeline (final clean run, 2016-2025)

**This notebook runs end-to-end on the first try.** It reads from your Drive's `data/` cache if present, falls back to a fresh SimFin download only if needed, builds the point-in-time panel, validates it, and saves outputs to `output/`.

**Window: 2016-Q1 → 2025-Q4 (40 quarters × 200 firms).**

All audit fixes baked in: SimFinId→ticker join, 4-quarter TTM, banks+insurance schemas, validation gate, horizon-validated forward returns.

## Step 1 — Setup (mount Drive, clone repo, install)

In [ ]:
# Mount Drive and create all working directories
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/cvm-research'
for sub in ('data', 'data/raw', 'output'):
    os.makedirs(f'{WORK_DIR}/{sub}', exist_ok=True)
print(f'Working dir: {WORK_DIR}')
print(f'  data/    contents: {sorted(os.listdir(f"{WORK_DIR}/data"))[:10]}')
print(f'  data/raw contents: {sorted(os.listdir(f"{WORK_DIR}/data/raw"))[:10]}')
print(f'  output/  contents: {sorted(os.listdir(f"{WORK_DIR}/output"))[:10]}')

In [ ]:
# Install + clone repo + load module
!pip install simfin pyarrow --quiet

import os, sys, subprocess
REPO_PATH = 'FelixDLanger/CVM-Research'
CLONE_DIR = '/content/cvm-research'

if os.path.exists(CLONE_DIR) and not os.path.exists(f'{CLONE_DIR}/cvm_research/pit_data.py'):
    subprocess.run(['rm', '-rf', CLONE_DIR])
if not os.path.exists(f'{CLONE_DIR}/cvm_research/pit_data.py'):
    r = subprocess.run(['git', 'clone', f'https://github.com/{REPO_PATH}.git', CLONE_DIR],
                       capture_output=True, text=True)
    print('clone rc:', r.returncode)
if CLONE_DIR not in sys.path:
    sys.path.insert(0, CLONE_DIR)

from cvm_research.pit_data import (
    quarter_ends, TickerIndex, ensure_ticker_column, _normalise_columns,
    build_pit_panel, attach_forward_returns, market_index_daily_returns,
)
print('✓ cvm_research module loaded')

## Step 2 — Configuration

Edit `START_YEAR`/`END_YEAR` to change the window. Validation thresholds halt the notebook if data is too sparse to analyse.

In [ ]:
START_YEAR = 2016
END_YEAR   = 2025

MIN_FUNDAMENTAL_COVERAGE = 0.50   # >=50% of rows must have real fundamentals
MIN_FORWARD_RETURN_COVERAGE = 0.40
REQUIRE_EARLY_YEAR_RETURNS = START_YEAR + 1

print(f'Study window: {START_YEAR}-Q1 to {END_YEAR}-Q4')
print(f'Validation gates: fund>={MIN_FUNDAMENTAL_COVERAGE:.0%}, fwd_ret>={MIN_FORWARD_RETURN_COVERAGE:.0%}')

## Step 3 — Load SimFin data (cache-first)

If raw parquets already exist in Drive's `data/raw/`, load from them. Otherwise download from SimFin and cache. Either way, you end up with the same DataFrames in memory.

**You may safely cancel SimFin after this step succeeds** — re-runs read from Drive.

In [ ]:
import pandas as pd
import glob

RAW_DIR = f'{WORK_DIR}/data/raw'
cached = glob.glob(f'{RAW_DIR}/*.parquet')
print(f'Raw cache: {len(cached)} parquet files in {RAW_DIR}')

if cached:
    print('Loading from Drive cache (no SimFin API calls)...')
    per_market = {}
    for path in cached:
        fname = os.path.basename(path).replace('.parquet','')
        # filenames are like 'us_income.parquet', 'de_balance.parquet'
        if '_' not in fname:
            continue
        mkt, kind = fname.split('_', 1)
        per_market.setdefault(mkt, {})[kind] = pd.read_parquet(path)
    for mkt, d in per_market.items():
        sizes = ' · '.join(f'{k}={len(v):,}' for k,v in d.items())
        print(f'  {mkt.upper()}: {sizes}')
else:
    print('Cache empty — downloading from SimFin (requires SIMFIN_API_KEY in Colab Secrets)...')
    from google.colab import userdata
    import simfin as sf
    sf.set_api_key(userdata.get('SIMFIN_API_KEY'))
    sf.set_data_dir(f'{WORK_DIR}/data')

    markets = ['us', 'de', 'uk', 'fr', 'jp', 'cn']
    def safe(loader, **kw):
        try:    return loader(**kw)
        except Exception: return pd.DataFrame()

    per_market = {}
    for mkt in markets:
        comp = safe(sf.load_companies, market=mkt)
        if len(comp) == 0:
            print(f'  {mkt.upper()}: SKIP'); continue
        inc = pd.concat([safe(sf.load_income, variant='quarterly', market=mkt),
                         safe(sf.load_income_banks, variant='quarterly', market=mkt),
                         safe(sf.load_income_insurance, variant='quarterly', market=mkt)], ignore_index=True)
        bal = pd.concat([safe(sf.load_balance, variant='quarterly', market=mkt),
                         safe(sf.load_balance_banks, variant='quarterly', market=mkt),
                         safe(sf.load_balance_insurance, variant='quarterly', market=mkt)], ignore_index=True)
        cf  = pd.concat([safe(sf.load_cashflow, variant='quarterly', market=mkt),
                         safe(sf.load_cashflow_banks, variant='quarterly', market=mkt),
                         safe(sf.load_cashflow_insurance, variant='quarterly', market=mkt)], ignore_index=True)
        prc = safe(sf.load_shareprices, variant='daily', market=mkt)
        per_market[mkt] = {'companies':comp, 'income':inc, 'balance':bal, 'cashflow':cf, 'prices':prc}
        print(f'  {mkt.upper()}: cos={len(comp)} inc={len(inc)} bal={len(bal)} px={len(prc):,}')

    # Cache to Drive immediately
    for mkt, d in per_market.items():
        for kind, df in d.items():
            if len(df):
                df.to_parquet(f'{RAW_DIR}/{mkt}_{kind}.parquet')
    print(f'\n✓ Cached to {RAW_DIR}/ — you may now cancel SimFin subscription.')

## Step 4 — Normalize columns + concatenate across markets

In [ ]:
# _normalise_columns handles SimFin's various index/column layouts (Ticker-indexed,
# MultiIndex, case variants) and produces uniform snake_case columns.
def norm(df):
    return _normalise_columns(df)

all_companies = pd.concat([norm(d.get('companies', pd.DataFrame())) for d in per_market.values()], ignore_index=True)
all_income    = pd.concat([norm(d.get('income',    pd.DataFrame())) for d in per_market.values()], ignore_index=True)
all_balance   = pd.concat([norm(d.get('balance',   pd.DataFrame())) for d in per_market.values()], ignore_index=True)
all_cashflow  = pd.concat([norm(d.get('cashflow',  pd.DataFrame())) for d in per_market.values()], ignore_index=True)
all_prices    = pd.concat([norm(d.get('prices',    pd.DataFrame())) for d in per_market.values()], ignore_index=True)

print(f'companies: {len(all_companies):,}  income: {len(all_income):,}  '
      f'balance: {len(all_balance):,}  prices: {len(all_prices):,}')
print(f'all_companies columns: {list(all_companies.columns)[:10]}')

## Step 5 — Join SimFinId → ticker (the bug we fixed permanently)

In [ ]:
all_income   = ensure_ticker_column(all_income,   all_companies)
all_balance  = ensure_ticker_column(all_balance,  all_companies)
all_cashflow = ensure_ticker_column(all_cashflow, all_companies)
all_prices   = ensure_ticker_column(all_prices,   all_companies)

print('Ticker join complete. Rows dropped as unmapped:')
for name, df in [('income',all_income), ('balance',all_balance),
                 ('cashflow',all_cashflow), ('prices',all_prices)]:
    print(f'  {name:8s}: {df.attrs.get("dropped_unmapped", 0)} dropped, {len(df):,} kept')

## Step 6 — Build per-ticker indexes

In [ ]:
income_idx   = TickerIndex(all_income,   date_col='publish_date')
balance_idx  = TickerIndex(all_balance,  date_col='publish_date')
cashflow_idx = TickerIndex(all_cashflow, date_col='publish_date')
price_idx    = TickerIndex(all_prices,   date_col='date')
print(f'Indexes built. Tickers: inc={len(income_idx._per_ticker)} '
      f'bal={len(balance_idx._per_ticker)} px={len(price_idx._per_ticker)}')

## Step 7 — Reference universe (200 tickers) + market proxy

In [ ]:
import urllib.request
TICKERS_URL = 'https://raw.githubusercontent.com/FelixDLanger/Concentric_Value_Model/main/tickers.csv'
tickers_path = f'{WORK_DIR}/tickers.csv'
urllib.request.urlretrieve(TICKERS_URL, tickers_path)

SECTOR_NORMALISE = {'HEALTH':'HC', 'IND':'INDUS', 'CONS_S':'CONS_D', 'REAL':'RE'}
ref = []
with open(tickers_path) as f:
    for line in f:
        if line.startswith('#') or line.startswith('ticker,'): continue
        p = line.strip().split(',')
        if len(p) >= 5:
            ref.append({'ticker':p[0], 'name':p[1], 'country':p[2], 'region':p[3],
                        'sector':SECTOR_NORMALISE.get(p[4], p[4])})
ref_df = pd.DataFrame(ref)
in_px = len(set(ref_df['ticker']) & set(all_prices['ticker']))
in_inc = len(set(ref_df['ticker']) & set(all_income['ticker']))
print(f'Universe: {len(ref_df)} tickers   In prices: {in_px}   In income: {in_inc}')

In [ ]:
# Market index proxy for beta
mkt_returns = market_index_daily_returns(price_idx, index_ticker='SPY')
if len(mkt_returns) > 100:
    market_proxy = 'SPY'
    print(f'✓ SPY proxy: {len(mkt_returns):,} daily returns')
else:
    market_proxy = 'EW_UNIVERSE'
    print('⚠ SPY unavailable — building equal-weight proxy')
    by_date = {}
    for t in ref_df['ticker']:
        sub = price_idx._per_ticker.get(t)
        if sub is None or len(sub) < 2: continue
        cc = 'adj_close' if 'adj_close' in sub.columns else 'close'
        s = sub.set_index('date')[cc].dropna().sort_index().pct_change().dropna()
        for d, r in s.items(): by_date.setdefault(d, []).append(r)
    mkt_returns = pd.Series({d: sum(v)/len(v) for d,v in by_date.items() if len(v)>=50}).sort_index()
    print(f'EW proxy: {len(mkt_returns):,} daily returns')
print(f'Market proxy: {market_proxy}')

## Step 8 — Build point-in-time panel

In [ ]:
QUARTERS = quarter_ends(START_YEAR, END_YEAR)
print(f'{len(QUARTERS)} quarter-ends: {QUARTERS[0].date()} → {QUARTERS[-1].date()}')

ticker_to_meta = {r['ticker']:{'sector':r['sector'], 'country':r['country'],
                               'name':r['name'], 'cap':None}
                  for _, r in ref_df.iterrows()}

print('Building panel (3-5 min)...')
panel = build_pit_panel(
    tickers=ref_df['ticker'].tolist(), quarter_dates=QUARTERS,
    income_idx=income_idx, balance_idx=balance_idx, cashflow_idx=cashflow_idx,
    price_idx=price_idx, ticker_to_meta=ticker_to_meta, market_index_returns=mkt_returns)
print(f'Panel: {len(panel):,} rows')
for col in ['pe','pb','market_cap','beta','roe','debt_equity']:
    if col in panel:
        n = panel[col].notna().sum()
        print(f'  {col:12s}: {n:5d} ({100*n/len(panel):4.1f}%)')

## Step 9 — Forward returns

In [ ]:
panel = attach_forward_returns(panel, price_idx, trading_days=252)
n_fwd = panel['forward_return_pct'].notna().sum()
print(f'Forward returns: {n_fwd:,} / {len(panel):,} ({100*n_fwd/len(panel):.1f}%)')
panel['year'] = pd.to_datetime(panel['as_of']).dt.year
print('\nBy year:')
print(panel.groupby('year')['forward_return_pct'].apply(lambda s: s.notna().sum()).to_string())

## Step 10 — 🚦 VALIDATION GATE

Halts the notebook if data is too sparse to analyse. If this prints `✅ PASSED`, the panel is good.

In [ ]:
errors = []

fund_cols = [c for c in ['pe','pb','roe','debt_equity','market_cap'] if c in panel.columns]
has_fund = panel[fund_cols].notna().any(axis=1).mean()
if has_fund < MIN_FUNDAMENTAL_COVERAGE:
    errors.append(f'Fundamental coverage {has_fund:.1%} < required {MIN_FUNDAMENTAL_COVERAGE:.0%}')

fwd_cov = panel['forward_return_pct'].notna().mean()
if fwd_cov < MIN_FORWARD_RETURN_COVERAGE:
    errors.append(f'Forward-return coverage {fwd_cov:.1%} < required {MIN_FORWARD_RETURN_COVERAGE:.0%}')

early = panel[panel['year'] <= REQUIRE_EARLY_YEAR_RETURNS]['forward_return_pct'].notna().sum()
if early == 0:
    errors.append(f'ZERO forward returns at/before {REQUIRE_EARLY_YEAR_RETURNS}')

dup = panel.duplicated(subset=['ticker','as_of']).sum()
if dup > 0:
    errors.append(f'{dup} duplicate (ticker, as_of) rows')

if errors:
    print('🛑 VALIDATION FAILED:')
    for e in errors: print(f'  ✗ {e}')
    raise RuntimeError('Panel failed validation — do NOT proceed.')
else:
    print('✅ VALIDATION PASSED')
    print(f'  fundamental coverage: {has_fund:.1%}')
    print(f'  forward-return coverage: {fwd_cov:.1%}')
    print(f'  earliest-year returns: {early} rows at/before {REQUIRE_EARLY_YEAR_RETURNS}')
    print(f'  duplicates: {dup}')

## Step 11 — Country macro (World Bank)

In [ ]:
import requests
WB = {'gdp_growth':'NY.GDP.MKTP.KD.ZG', 'inflation':'FP.CPI.TOTL.ZG'}
CMAP = {'US':'USA','DE':'DEU','NL':'NLD','FR':'FRA','DK':'DNK','CH':'CHE','IT':'ITA',
        'ES':'ESP','BE':'BEL','GB':'GBR','JP':'JPN','TW':'TWN','KR':'KOR','CN':'CHN',
        'HK':'HKG','IN':'IND','SG':'SGP'}
recs = []
for c2, c3 in CMAP.items():
    for lab, ind in WB.items():
        try:
            r = requests.get(f'https://api.worldbank.org/v2/country/{c3}/indicator/{ind}',
                             params={'format':'json', 'date':f'{START_YEAR}:{END_YEAR}', 'per_page':'30'},
                             timeout=15)
            d = r.json()
            if isinstance(d, list) and len(d) > 1:
                for rec in d[1] or []:
                    if rec.get('value') is not None:
                        recs.append({'country':c2, 'year':int(rec['date']),
                                     'indicator':lab, 'value':float(rec['value'])})
        except Exception as e:
            pass
if recs:
    macro_wide = (pd.DataFrame(recs)
                  .pivot_table(index=['country','year'], columns='indicator', values='value')
                  .reset_index())
    panel = panel.merge(macro_wide, on=['country','year'], how='left')
    panel = panel.rename(columns={'gdp_growth':'macro_gdp', 'inflation':'macro_inflation'})
    print(f'Macro merged: {len(macro_wide)} country-years')
else:
    macro_wide = pd.DataFrame()
    print('Macro skipped (no data fetched)')

## Step 12 — Enrich panel with ownership + ESG (US firms only)

Pulls insider %, institutional %, and ESG risk from yfinance for US tickers. Non-US tickers retain sector baseline on D6/D7/D9.

**Methodological note:** ownership values are point-in-time snapshots held constant across the study period. The cross-sectional ranking (which firms have more institutional ownership) is the variation used by the quartile analysis. Within-firm time-series variation in ownership is NOT captured here.

ESG is attached only to observations within 18 months of today, to limit look-ahead bias from using current Sustainalytics ratings on historical data.


In [ ]:
from cvm_research.ownership_enrichment import enrich_panel_with_ownership

# Enrich US firms with yfinance ownership + ESG snapshot data.
# This takes 3-5 minutes (one yfinance call per US ticker).
panel = enrich_panel_with_ownership(
    panel,
    us_only=True,         # non-US data is unreliable for free
    attach_esg=True,      # Yahoo Sustainability (US large-caps mostly)
    esg_recency_months=18 # ESG only on recent obs (look-ahead discipline)
)


## Step 13 — Save outputs

In [ ]:
panel.to_parquet(f'{WORK_DIR}/output/panel_pit_{START_YEAR}_{END_YEAR}.parquet')
if len(macro_wide):
    macro_wide.to_parquet(f'{WORK_DIR}/output/macro_history.parquet')
panel.to_csv(f'{WORK_DIR}/output/panel_pit_{START_YEAR}_{END_YEAR}.csv', index=False)

import json

meta = {
    'build': 'phase2-v2.3-clean',
    'window': f'{START_YEAR}-{END_YEAR}',
    'n_quarters': len(QUARTERS),
    'n_obs': len(panel),
    'n_forward_returns': int(panel['forward_return_pct'].notna().sum()),
    'market_proxy': market_proxy,
    'fundamental_coverage': float(has_fund),
    'forward_return_coverage': float(fwd_cov),
}
with open(f'{WORK_DIR}/output/panel_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)
print('Saved:')
print(f'  panel_pit_{START_YEAR}_{END_YEAR}.parquet')
print(f'  panel_pit_{START_YEAR}_{END_YEAR}.csv')
print(f'  panel_metadata.json')
print(f'  macro_history.parquet')
print('\n→ Proceed to 02_pca_analysis.ipynb')

## Summary

Validated 2016-2025 panel ready for analysis. Next: nb02 (PCA), then nb03 (regression).